**Investments: Theory and Data Analysis**, Bates, Boyer, and Fletcher


# Chapter 6: Alpha Vantage Stock Data with `farms`

This notebook introduces the `farms` Alpha Vantage loader for data on public equities.

## Learning objectives

By the end of this notebook, you should be able to:

- Set an Alpha Vantage API key as a Windows user environment variable without placing it in a notebook.
- Use `farms.load_alpha_vantage()` to load data with one ticker.
- Distinguish raw closing prices from adjusted closing prices.
- Calculate simple returns from adjusted closing prices.
- Recognize API-rate-limit and credential-management issues.

## Data access

The examples require an Alpha Vantage API key and an internet connection. To get a key, visit the [Alpha Vantage API key page](https://www.alphavantage.co/support/#api-key), enter your email address, and follow the instructions to receive your free key. Copy the key exactly as provided.

### Windows setup
For permanent Windows setup, you will need to run a command in `PowerShell`, Windows' command-line and scripting program. To open a `PowerShell` session, select the Windows Start menu, type `PowerShell`, and choose `Windows PowerShell` or `PowerShell` from the results. Then run the following command by copying and pasting at the `PowerShell` prompt. Replace only the placeholder value `your-alpha-vantage-key` with your Alpha Vantage key:

```powershell
[Environment]::SetEnvironmentVariable(
    "ALPHAVANTAGE_API_KEY",
    "your-alpha-vantage-key",
    "User"
)
```

The `User` scope saves the variable for future Windows sessions. Close and reopen this notebook, or restart the kernel so the notebook process can see the new variable. Do not place the key directly in a code cell.

### Google Colab setup

Colab runs in a temporary cloud runtime, so the Windows PowerShell environment variable is not available there. Colab **Secrets** are the recommended permanent setup for Colab: the secret is stored separately from the notebook and remains available after the runtime resets.

To set it up, open the **Secrets** panel using the key icon in the left sidebar, add a secret named `ALPHAVANTAGE_API_KEY`, paste your Alpha Vantage key as its value, and enable notebook access for that secret.

A Colab Secret does not become a Windows environment variable. The notebook retrieves it directly with `userdata.get('ALPHAVANTAGE_API_KEY')`. Secrets are tied to your Colab account and are not shared automatically with people who open a shared notebook; each collaborator must add their own secret.

The Colab and Windows setup methods keep the API key outside the notebook. The installation cell force-reinstalls `farms==0.1.36` from PyPI so the notebook uses the Alpha Vantage loader with Ken French factor support. If an older `farms` version was already imported, restart the kernel or runtime after installation before running the remaining cells.

In [ ]:
%pip install -q farms

import os

import farms as fm
import matplotlib.pyplot as plt
import pandas as pd

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)

### Read the API key securely

The `farms.get_alpha_vantage_api_key()` function reads your API key. It supports both Windows and Google Colab environments. The function never prints the key.

In [ ]:
api_key = fm.get_alpha_vantage_api_key()

## Use the generalized loader

Use `fm.load_alpha_vantage()` to retrieve one selected field for either a single ticker or multiple tickers. The function supports monthly and weekly data.

With a free Alpha Vantage account, each ticker requires a separate API request. Multi-ticker requests may encounter request limits. For reliable free-tier use, request one ticker at a time, wait between requests, and then merge the results by date.

**Required inputs**

- `symbol`: one ticker string, such as `"MSFT"`, or an iterable of ticker strings, such as `["MSFT", "AAPL"]`.
- `api_key`: your Alpha Vantage API key, loaded securely in the preceding cell.
- `frequency`: `"monthly"` or `"weekly"`.
- `field`: `"open"`, `"high"`, `"low"`, `"close"`, or `"returns"`. `"returns"` is the decimal percentage change in adjusted close.

**Optional inputs**

- `start_date`: lower inclusive date bound; default is `None` (all available data).
    - Use the format `YYYY-MM-DD` for weekly data,.
    - Use `YYYY-MM` for monthly data and `YYYY-MM-DD` for weekly data.
- `end_date`: upper inclusive date bound; default is `None` (all available data).
    - Use the same format as `start_date`.
- `timeout`: request timeout in seconds; default is `30`.
- `max_retries`: number of retries for transient failures and rate-limit responses; default is `3`.
- `backoff_factor`: starting delay between retries; default is `1.0`.
- `include_factors`: factor set to merge from the Ken French library; default is `"market"`. Choose `"market"`, `"ff3"`, `"ff5"`, or `"none"`.
- `session`: an optional `requests.Session`; default is `None`, which uses the standard requests call.

## Load MSFT returns and market returns

The generalized loader can return MSFT's monthly total return together with the Fama-French market excess return (`ff_mkt_rf`) and risk-free rate (`ff_rf`). The total market return is the sum of those two factor columns.

In [ ]:
msft = fm.load_alpha_vantage(
    symbol='MSFT',
    api_key=api_key,
    frequency='monthly',
    field='returns',
    include_factors='market',
    start_date='2021-01',
    end_date='2025-12',
)

msft.head()

### Summary Statistics
The `describe()` method produces summary statistics for the MSFT total return series.

In [ ]:
msft["Return"].describe()

## Construct total market return

`Return` is MSFT's decimal total return calculated from adjusted close. `ff_mkt_rf` is the market excess return and `ff_rf` is the risk-free rate. Adding them produces the Fama-French total market return.

In [ ]:
analysis = msft.rename(columns={'Return': 'msft_total_return'}).copy()
analysis['market_total_return'] = analysis['ff_mkt_rf'] + analysis['ff_rf']
analysis = analysis[['msft_total_return', 'market_total_return']].dropna()
analysis.head()

### Create Scatter Plot
The scatter plot shows the relationship between MSFT's total return and the total market return. Each

In [ ]:
ax = analysis.plot.scatter(
    x='market_total_return',
    y='msft_total_return',
    figsize=(8, 6),
    alpha=0.75,
    title='MSFT Total Return vs. Total Market Return',
)
ax.set_xlabel('Total market return')
ax.set_ylabel('MSFT total return')
plt.show()

## Common mistakes and security reminders

- Do not hard-code `ALPHAVANTAGE_API_KEY` in a notebook or commit it to Git.
- If the environment variable is set after Jupyter starts, restart the Jupyter server or kernel.
- Use `YYYY-MM` for monthly bounds and `YYYY-MM-DD` for weekly bounds.
- `Close` is the unadjusted closing price; `Adjusted Close` reflects historical split and dividend adjustments.
- Alpha Vantage can return rate-limit or informational messages instead of time-series data. The `farms` loaders raise clear exceptions for those responses.
- Daily Alpha Vantage data is not supported by these examples because the adjusted daily endpoint requires premium access.

## Try it

1. Replace `MSFT` with another supported symbol and compare its monthly returns with the total market return.
2. Change the monthly date range and inspect the `PeriodIndex`.
3. Try `include_factors='ff3'` or `include_factors='ff5'` and inspect the additional factor columns.
4. Use `include_factors='none'` when you only need Alpha Vantage data.
5. Explain why storing an API key in a user environment variable is safer than placing it in a notebook cell.